# Session 3: from one stress to a real load history
**A. Real-world problem.** A bracket on a machine does not see a neat sine wave. It sees
vibration, start-ups and now and then a hard knock. How many "shifts" will it last, and how
often must we inspect it for cracks? Today we (1) count the cycles hidden in a messy history
(*rainflow counting*), (2) add up their damage (*Miner's rule*), (3) grow a crack with
*Paris' law*, (4) see how widely real lives scatter. All numbers are illustrative; this is a
teaching model and never a method for certifying a real or safety-critical part.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

SU, SE = 600.0, 300.0                                  # MPa, illustrative steel
B = -np.log10(0.9 * SU / SE) / 3.0                     # Basquin exponent b
A = (0.9 * SU) ** 2 / SE                               # Basquin coefficient, MPa

## Step 1: rainflow counting
A history is a list of stresses in time order. Only the *turning points* (peaks and valleys,
called reversals) matter. The standard three-point rule (ASTM E1049) keeps a stack of turning
points. When the newest range X is at least as large as the previous range Y, the range Y is
a closed loop: it counts as one *full cycle* (or a *half cycle* if it contains the start of
the history). What is left over at the end is counted as half cycles. A useful check: the sum
of 2 * count * range over all counted cycles equals the total up-and-down travel of the history.

In [ ]:
def reversals(x):
    """Turning points of a series (drops repeats and points in the middle of a slope)."""
    x = np.asarray(x, dtype=float)
    keep = [x[0]]
    for v in x[1:]:
        if v == keep[-1]:
            continue
        if len(keep) >= 2 and (keep[-1] - keep[-2]) * (v - keep[-1]) > 0:
            keep[-1] = v                 # still going the same way: extend
        else:
            keep.append(v)
    return np.array(keep)


def rainflow(x):
    """ASTM E1049 three-point rainflow. Returns a list of (range, mean, count), count 1.0 or 0.5."""
    stack, out = [], []
    for p in reversals(x):
        stack.append(p)
        while len(stack) >= 3:
            xr = abs(stack[-1] - stack[-2])
            yr = abs(stack[-2] - stack[-3])
            if xr < yr:
                break
            mean = 0.5 * (stack[-2] + stack[-3])
            if len(stack) == 3:                  # range Y contains the start: half cycle
                out.append((yr, mean, 0.5))
                stack.pop(0)
            else:                                # closed loop: full cycle
                out.append((yr, mean, 1.0))
                del stack[-3:-1]
    for i in range(len(stack) - 1):              # residue: half cycles
        out.append((abs(stack[i + 1] - stack[i]), 0.5 * (stack[i + 1] + stack[i]), 0.5))
    return out

## Step 2: Miner's rule
If a block of `n_i` cycles at some level has a life of `N_i` cycles at that level alone, it
uses up the fraction `n_i / N_i` of the life. Failure is predicted when the sum reaches 1:

    D = sum( n_i / N_i ) = 1

This *linear damage rule* is standard but rough: it ignores the order of the loads (a few big
cycles early change how later small cycles act) and in tests the sum at failure scatters
roughly between 0.3 and 3. Each cycle's mean stress is handled with Goodman (Session 2).

In [ ]:
def cycle_life(amplitude, mean=0.0, below="extend"):
    """Cycles to failure of one rainflow cycle (amplitude and mean in MPa).
    below='extend' keeps the Basquin line under S_e (conservative); below='ignore' gives inf."""
    if mean >= SU:
        return 1.0
    eq = amplitude / (1.0 - mean / SU)
    if below == "ignore" and eq <= SE:
        return np.inf
    if eq <= 0:
        return np.inf
    return max(1.0, float((eq / A) ** (1.0 / B)))


def miner_damage(cycles, below="extend"):
    """Total damage D of a rainflow list of (range, mean, count). Failure is predicted at D = 1."""
    return float(sum(c / cycle_life(r / 2.0, m, below) for r, m, c in cycles))


def damage_shares(cycles, below="extend"):
    """Damage of each rainflow entry, as a numpy array in the same order."""
    return np.array([c / cycle_life(r / 2.0, m, below) for r, m, c in cycles])


def damage_share_above(cycles, amplitude, below="extend"):
    """Fraction of the total damage that comes from cycles whose amplitude (range/2) exceeds `amplitude`."""
    sh = damage_shares(cycles, below)
    big = np.array([r / 2.0 > amplitude for r, _, _ in cycles])
    return float(sh[big].sum() / sh.sum())

## Step 3: crack growth (Paris' law)
Once a crack exists, every cycle makes it longer by `da/dN = C * (delta_K)^m`, with the
stress-intensity range `delta_K = Y * delta_sigma * sqrt(pi * a)` (MPa sqrt(m) when the
stress range is in MPa and the length a is in m). Y is a shape factor (1.12 for a small edge
crack), C and m are material constants (typical steel: C = 6.9e-12 and m = 3 for da/dN in m
per cycle). The crack becomes critical when the largest K reaches the fracture toughness K_Ic:
`a_c = (K_Ic / (Y * sigma_max * sqrt(pi)))^2`. The cycles from a0 to a_c follow from
integrating `dN = da / (C delta_K^m)`; for constant Y and m not equal to 2 there is a closed
form, which we use to test the numerical integral.

In [ ]:
C_PARIS, M_PARIS, Y_EDGE, KIC = 6.9e-12, 3.0, 1.12, 50.0


def delta_k(dsig, a, y=Y_EDGE):
    """Stress-intensity range, MPa sqrt(m), for stress range dsig (MPa) and crack length a (m)."""
    return y * dsig * np.sqrt(np.pi * a)


def critical_crack(sigma_max, kic=KIC, y=Y_EDGE):
    """Crack length (m) at which K_max = K_Ic."""
    return (kic / (y * sigma_max * np.sqrt(np.pi))) ** 2


def paris_closed_form(a0, ac, dsig, c=C_PARIS, m=M_PARIS, y=Y_EDGE):
    """Cycles from a0 to ac for constant Y and m != 2."""
    k = 1.0 - m / 2.0
    return (ac ** k - a0 ** k) / (c * y ** m * dsig ** m * np.pi ** (m / 2.0) * k)


def paris_numeric(a0, ac, dsig, c=C_PARIS, m=M_PARIS, y=Y_EDGE, n=4000):
    """Cycles from a0 to ac by trapezoid integration of a/(C dK^m) over u = ln a. y may be a function of a."""
    u = np.linspace(np.log(a0), np.log(ac), n + 1)
    a = np.exp(u)
    yy = y(a) if callable(y) else y
    f = a / (c * delta_k(dsig, a, yy) ** m)
    return float(np.sum(0.5 * (f[1:] + f[:-1]) * np.diff(u)))


def paris_curve(a0, ac, dsig, c=C_PARIS, m=M_PARIS, y=Y_EDGE, n=400):
    """(cycles, crack length) arrays for plotting the growth of the crack."""
    u = np.linspace(np.log(a0), np.log(ac), n + 1)
    a = np.exp(u)
    f = a / (c * delta_k(dsig, a, y) ** m)
    cyc = np.concatenate([[0.0], np.cumsum(0.5 * (f[1:] + f[:-1]) * np.diff(u))])
    return cyc, a

## Step 4: scatter and sensitivity
Identical parts at the same stress do not fail at the same time. Life N is roughly *lognormal*:
log10(N) is a normal (bell-curve) number around the predicted value. We use a standard
deviation of 0.3 in log10(N), an illustrative value. And because life goes as stress to the
power 1/b, a small stress change has a big effect: a factor `r` in stress changes life by `r^(1/b)`.

In [ ]:
def lognormal_lives(n_median, sigma_log10=0.3, size=10000, seed=42):
    """Random lives with median n_median and spread sigma_log10 (in log10 N). Seeded."""
    rng = np.random.default_rng(seed)
    return n_median * 10.0 ** rng.normal(0.0, sigma_log10, size)


def life_ratio_for_stress_change(ratio, exponent=B):
    """New life / old life when the stress amplitude is multiplied by `ratio` (Basquin)."""
    return ratio ** (1.0 / exponent)

## Step 5: a bracket and a measured-style load history
The function below builds a synthetic "measured" history of one shift: load in newtons at the
bracket's hot spot, converted to stress with 0.1 MPa per newton (this factor includes the
notch effect; an assumption). It has steady vibration, 10 start-ups and 6 rare hard knocks.

In [ ]:
K_STRESS = 0.1      # MPa per newton at the hot spot (assumed)


def bracket_history(seed=7, n=4000, n_knocks=6):
    """Hot-spot stress (MPa) at 4000 equally spaced instants of one shift."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    f = 1000 + 700 * np.sin(2 * np.pi * t / 37) * (1 + 0.4 * np.sin(2 * np.pi * t / 500)) / 1.4 + rng.normal(0, 60, n)
    pulse = np.array([0.1, 0.5, 1, 1, 0.5, 0.1])
    for k in range(250, n - 10, 400):
        f[k:k + 6] += 3000 * pulse
    for k in np.linspace(300, n - 300, n_knocks).astype(int):
        f[k - 2:k + 3] = [-500, -1200, 4500, -1200, -500]
    return K_STRESS * f


def effective_range(rng_, mean):
    """Part of a cycle's range that is in tension (a crack is closed in compression)."""
    smax, smin = mean + rng_ / 2.0, mean - rng_ / 2.0
    return max(smax, 0.0) - max(smin, 0.0)


def equivalent_range(cycles, m=M_PARIS):
    """Constant stress range with the same Paris growth as the whole list (m-th power mean)."""
    num = sum(c * effective_range(r, mn) ** m for r, mn, c in cycles)
    den = sum(c for r, mn, c in cycles)
    return float((num / den) ** (1.0 / m))


def inspection_interval_shifts(stress, a_detect=0.5e-3, safety=3.0):
    """Shifts between inspections: (shifts for a crack to grow from a_detect to critical) / safety."""
    cyc = rainflow(stress)
    dsig = equivalent_range(cyc)
    ac = critical_crack(stress.max())
    n_cycles = paris_closed_form(a_detect, ac, dsig)
    per_shift = sum(c for r, mn, c in cyc)
    return n_cycles / per_shift / safety

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    seq = [-2, 1, -3, 5, -1, 3, -4, 4, -2]
    print("ASTM-style test sequence:", [(float(r), float(m), c) for r, m, c in rainflow(seq)])

    s = bracket_history()
    cyc = rainflow(s)
    d_ext, d_ign = miner_damage(cyc), miner_damage(cyc, "ignore")
    shares = damage_shares(cyc)
    cnt_big = sum(c for r, _, c in cyc if r / 2 > 250)
    print(f"cycles above 250 MPa amplitude: {cnt_big} of {sum(c for _, _, c in cyc)}, damage share {damage_share_above(cyc, 250):.3f}")
    print(f"no knocks: life {1 / miner_damage(rainflow(bracket_history(n_knocks=0))):,.0f} shifts")
    print(f"cycles counted {sum(c for _, _, c in cyc):.1f}; D per shift {d_ext:.3e} -> life {1 / d_ext:,.0f} shifts")
    print(f"ignoring cycles below S_e: D = {d_ign:.3e} -> {1 / d_ign:,.0f} shifts; ")
    print(f"inspection interval {inspection_interval_shifts(s):.1f} shifts")

    fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
    axs[0].plot(s, lw=0.6)
    axs[0].set_xlabel("time step within one shift")
    axs[0].set_ylabel("hot-spot stress (MPa)")
    axs[0].set_title("Synthetic load history of one shift")
    axs[0].grid(alpha=0.3)
    amps = np.array([r / 2 for r, _, _ in cyc])
    axs[1].scatter(amps, shares / shares.sum(), s=10)
    axs[1].set_yscale("log")
    axs[1].set_xlabel("cycle amplitude (MPa)")
    axs[1].set_ylabel("share of total damage")
    axs[1].set_title("A few big cycles carry almost all the damage")
    axs[1].grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(OUT / "history_damage.png", dpi=120)

    a0, dsig = 1e-3, 100.0
    ac = critical_crack(dsig)
    n_cf, n_num = paris_closed_form(a0, ac, dsig), paris_numeric(a0, ac, dsig)
    cyc2, a = paris_curve(a0, ac, dsig)
    fig2, ax2 = plt.subplots(figsize=(6.2, 4.0))
    ax2.plot(cyc2, a * 1000)
    ax2.set_xlabel("load cycles")
    ax2.set_ylabel("crack length (mm)")
    ax2.set_title("Paris' law: slow at first, then faster and faster")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "crack_growth.png", dpi=120)
    print(f"a_c = {ac * 1000:.1f} mm; closed form {n_cf:,.0f}; numeric {n_num:,.0f} cycles")

    lives = lognormal_lives(1e5)
    fig3, ax3 = plt.subplots(figsize=(6.2, 4.0))
    ax3.hist(np.log10(lives), bins=50)
    ax3.set_xlabel("log10 of cycles to failure")
    ax3.set_ylabel("number of parts (of 10000)")
    ax3.set_title("Scatter: same stress, very different lives")
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "scatter.png", dpi=120)
    p5, p95 = np.percentile(lives, [5, 95])
    print(f"5th to 95th percentile ratio {p95 / p5:.1f}")
    print(f"10 percent more stress: life x {life_ratio_for_stress_change(1.1):.3f}")

## Where this stops
Miner's rule ignores load order, rainflow ignores timing, Paris' law uses a fixed Y and no
crack closure, and the scatter width is only an example. Treat the outputs as estimates
for learning, never as a certificate for a real part.

## Coding tasks
1. **Easy.** Run `rainflow([0, 5, 0, 5, 0])` by hand first (how many full and half cycles?) and then in Python.
2. **Stretch.** Change `n_knocks` in `bracket_history` and print the life in shifts for 0, 4 and 10 knocks. Which number of knocks first drops the life below 4,000 shifts?
3. **Explain.** Explain why the closed form for Paris' law fails for m = 2 and what the life is for m = 2 instead (hint: the integral of da/a).

## Assessment
- **A. Conceptual.** Give two reasons why Miner's rule can be wrong, and say whether ignoring cycles below the endurance limit is safe when large overloads are also present.
- **B. Numerical.** An edge crack (Y = 1.12) in a part sees a stress range of 80 MPa (zero to maximum). Using C = 6.9e-12, m = 3 and K_Ic = 50 MPa sqrt(m), find delta K for a0 = 2 mm, the critical length and the life in cycles.
- **C. Graph and simulation interpretation.** In the scatter histogram the 5th percentile life is 31,000 cycles and the 95th is 311,000 for a median of 100,000. What does the spread say about testing one part, and what ratio of 95th to 5th percentile do you read off?
- **D. Design and optimisation challenge.** Design: the bracket's life is 4,998 shifts with 6 hard knocks per shift. A customer needs at least 4,000 shifts. How many knocks per shift may the machine produce at most? And the inspection method can find a crack of 1 mm instead of 0.5 mm: what happens to the interval, and what if it can only find 2 mm?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # --- rainflow: hand-traced test sequence and known cases
    seq = [-2, 1, -3, 5, -1, 3, -4, 4, -2]
    got = sorted((float(r), float(c)) for r, m, c in L.rainflow(seq))
    want = sorted([(3.0, 0.5), (4.0, 0.5), (4.0, 1.0), (8.0, 0.5), (9.0, 0.5), (8.0, 0.5), (6.0, 0.5)])
    checks.append(("rainflow of -2,1,-3,5,-1,3,-4,4,-2 gives one full cycle of range 4 and half cycles 3, 4, 8, 9, 8, 6", got == want))
    t = np.arange(0, 401)
    sine = 100.0 * np.cos(2 * np.pi * t / 40.0)           # exactly 10 full cycles, range 200
    cyc = L.rainflow(sine)
    checks.append(("constant-amplitude sine with 10 cycles gives 10 full cycles of range 200",
                   abs(sum(c for r, m, c in cyc) - 10.0) < 1e-9 and all(abs(r - 200.0) < 1e-6 for r, m, c in cyc)))
    checks.append(("a single rise 0 to 5 is one half cycle of range 5", [(float(r), c) for r, m, c in L.rainflow([0, 1, 3, 5])] == [(5.0, 0.5)]))
    rng = np.random.default_rng(3)
    walk = np.cumsum(rng.normal(size=300))
    rev = L.reversals(walk)
    cyc = L.rainflow(walk)
    total = float(np.abs(np.diff(rev)).sum())
    checks.append(("invariant: sum of 2 * count * range equals the total travel of the history (random walk)",
                   abs(sum(2 * c * r for r, m, c in cyc) - total) < 1e-9))
    checks.append(("edge: a constant history has no cycles", L.rainflow([3.0, 3.0, 3.0]) == []))

    # --- Miner
    n400, n350 = L.cycle_life(400), L.cycle_life(350)
    d1, d2 = 1e4 / n400, 5e4 / n350
    checks.append((f"worked example: 10,000 cycles at 400 MPa use 0.294 and 50,000 at 350 MPa use 0.306 (got {d1:.3f}, {d2:.3f}); sum 0.600",
                   abs(d1 - 0.294) < 5e-4 and abs(d2 - 0.306) < 5e-4 and abs(d1 + d2 - 0.600) < 5e-4))
    checks.append(("worked example: the remaining 0.400 allows about 13,600 more cycles at 400 MPa", abs(0.4 * n400 - 13607) < 5))
    half = [(800.0, 0.0, n400 / 2), (700.0, 0.0, n350 / 2)]
    checks.append(("Miner: half the life at each of two levels sums to D = 1", abs(L.miner_damage(half) - 1.0) < 1e-9))
    checks.append(("Miner: a full-life block of one level gives D = 1", abs(L.miner_damage([(800.0, 0.0, n400)]) - 1.0) < 1e-9))
    checks.append(("a tensile mean stress adds damage", L.cycle_life(250, 100) < L.cycle_life(250, 0)))
    checks.append(("below S_e: extend mode gives finite life, ignore mode gives infinite", np.isfinite(L.cycle_life(250)) and L.cycle_life(250, 0, "ignore") == np.inf))
    checks.append(("edge: mean at S_u gives a life of 1 cycle (no division by zero)", L.cycle_life(10, 600.0) == 1.0))

    # --- Paris' law
    a0, dsig = 1e-3, 100.0
    ac = L.critical_crack(dsig)
    dk0 = L.delta_k(dsig, a0)
    checks.append((f"worked example: delta_K at a0 = 1 mm is 6.28 MPa sqrt(m) (got {dk0:.2f}) and a_c = 63.4 mm (got {ac * 1000:.1f})",
                   abs(dk0 - 6.28) < 0.01 and abs(ac * 1000 - 63.4) < 0.05))
    ncf = L.paris_closed_form(a0, ac, dsig)
    nnum = L.paris_numeric(a0, ac, dsig)
    checks.append((f"worked example: closed form {ncf:,.0f} cycles from 1 mm to 63.4 mm (about 1.02 million); numeric agrees",
                   abs(ncf - 1024557) < 5 and abs(nnum / ncf - 1) < 1e-6))
    errs = []
    for m in (2.5, 3.0, 4.0):
        for a_0 in (0.5e-3, 2e-3):
            errs.append(abs(L.paris_numeric(a_0, 0.02, 150.0, m=m, c=1e-12) / L.paris_closed_form(a_0, 0.02, 150.0, m=m, c=1e-12) - 1))
    checks.append(("numeric integral matches the closed form for m = 2.5, 3, 4 (relative error below 1e-6)", max(errs) < 1e-6))
    m2 = L.paris_numeric(a0, 0.02, 100.0, m=2.0)
    ln_form = np.log(0.02 / a0) / (L.C_PARIS * L.Y_EDGE ** 2 * 100.0 ** 2 * np.pi)
    checks.append(("edge: m = 2 (closed form undefined) still integrates and matches the logarithmic formula", abs(m2 / ln_form - 1) < 1e-6))
    checks.append(("Y given as a function of a gives the same answer when it is constant", abs(L.paris_numeric(a0, ac, dsig, y=lambda a: np.full_like(a, 1.12)) / nnum - 1) < 1e-12))
    cy, aa = L.paris_curve(a0, ac, dsig)
    checks.append(("crack curve starts at 0 cycles and a0, ends at the closed-form life and a_c, and accelerates",
                   cy[0] == 0 and abs(cy[-1] / ncf - 1) < 1e-4 and abs(aa[-1] - ac) < 1e-12 and np.all(np.diff(cy) > 0)))
    r_same = L.paris_closed_form(a0, ac, 110.0) / ncf
    r_paris = L.paris_closed_form(a0, L.critical_crack(110.0), 110.0) / ncf
    checks.append((f"Paris: 10 percent more stress range cuts life to 1.1^-3 = 0.751 at the same a_c (got {r_same:.3f}), a cut of 25 percent; with the smaller a_c it is {r_paris:.3f}",
                   abs(r_same - 1.1 ** -3) < 1e-9 and abs(r_paris - 0.741) < 0.002))

    # --- scatter and sensitivity
    l1, l2 = L.lognormal_lives(1e5), L.lognormal_lives(1e5)
    checks.append(("seeded Monte Carlo is repeatable", bool(np.array_equal(l1, l2))))
    p1, p5, p95 = np.percentile(l1, [1, 5, 95])
    checks.append((f"median of 10,000 simulated lives is within 3% of 100,000 (got {np.median(l1):,.0f})", abs(np.median(l1) / 1e5 - 1) < 0.03))
    checks.append((f"5th to 95th percentile ratio is about 10 (got {p95 / p5:.1f}); theory 10^(2*1.645*0.3) = 9.7", abs(p95 / p5 - 9.7) < 0.7))
    checks.append((f"the 1st percentile life is about 0.2 of the median (got {p1 / 1e5:.2f})", abs(p1 / 1e5 - 0.2) < 0.02))
    r = L.life_ratio_for_stress_change(1.1)
    checks.append((f"S-N: 10 percent more stress multiplies life by 1.1^(1/b) = 0.326 (got {r:.3f}), a cut of 67 percent; note 1.1^b would be 0.992 (wrong)",
                   abs(r - 0.326) < 0.001 and abs(1.1 ** L.B - 0.992) < 0.001))

    # --- bracket
    s = L.bracket_history()
    cyc = L.rainflow(s)
    total_cycles = sum(c for r_, m_, c in cyc)
    d = L.miner_damage(cyc)
    checks.append((f"bracket: stress ranges from -120 to 450 MPa (got {s.min():.0f} to {s.max():.0f}); {total_cycles:.1f} cycles counted per shift",
                   abs(s.min() + 120) < 1 and abs(s.max() - 450) < 1 and abs(total_cycles - 920.5) < 1e-9))
    checks.append((f"bracket: damage per shift 2.0e-4, life about 5,000 shifts (got {d:.3e}, {1 / d:,.0f})", abs(d - 2.001e-4) < 1e-6 and abs(1 / d - 4998) < 5))
    d_ign = L.miner_damage(cyc, "ignore")
    checks.append((f"bracket: ignoring cycles below S_e gives about 5,050 shifts (got {1 / d_ign:,.0f}); the small cycles hardly matter", abs(1 / d_ign - 5046) < 5))
    share = L.damage_share_above(cyc, 250)
    n_big = sum(c for r_, m_, c in cyc if r_ / 2 > 250)
    checks.append((f"failure case: {n_big} of {total_cycles:.1f} cycles (about 1 percent) above 250 MPa carry {share:.1%} of the damage", n_big == 9.5 and abs(share - 0.967) < 0.001))
    life0 = 1 / L.miner_damage(L.rainflow(L.bracket_history(n_knocks=0)))
    checks.append((f"failure case: without the 6 hard knocks the life is 56,600 shifts, 11 times longer (got {life0:,.0f}, {life0 * d:.1f})", abs(life0 - 56607) < 10 and abs(life0 * d - 11.3) < 0.1))
    dsig_eq = L.equivalent_range(cyc)
    ac_b = L.critical_crack(s.max())
    iv = L.inspection_interval_shifts(s)
    checks.append((f"inspection: equivalent range {dsig_eq:.1f} MPa, a_c = {ac_b * 1000:.2f} mm, interval {iv:.1f} shifts (use 250)",
                   abs(dsig_eq - 112.6) < 0.1 and abs(ac_b * 1000 - 3.13) < 0.01 and abs(iv - 252.1) < 0.2))
    passing = [k for k in range(0, 15) if 1 / L.miner_damage(L.rainflow(L.bracket_history(n_knocks=k))) >= 4000]
    l8 = 1 / L.miner_damage(L.rainflow(L.bracket_history(n_knocks=8)))
    l12 = 1 / L.miner_damage(L.rainflow(L.bracket_history(n_knocks=12)))
    checks.append((f"mini-project: for a 4,000 shift requirement the most knocks per shift allowed is {max(passing)}; 8 knocks give {l8:,.0f}, 12 give {l12:,.0f}",
                   max(passing) == 7 and abs(l8 - 3799) < 5 and abs(l12 - 2781) < 5))
    checks.append(("mini-project: more knocks shorten the inspection interval (475 shifts with none, 174 with 12)",
                   abs(L.inspection_interval_shifts(L.bracket_history(n_knocks=0)) - 475.4) < 0.5 and abs(L.inspection_interval_shifts(L.bracket_history(n_knocks=12)) - 173.7) < 0.5))
    checks.append(("roadmap: the shipped Miner code treats the order of loads as irrelevant (reordered blocks give the same D)",
                   abs(L.miner_damage(half) - L.miner_damage(half[::-1])) < 1e-12))
    # extra numbers used in the text
    ac80 = L.critical_crack(80.0)
    checks.append((f"Section F: edge crack, 80 MPa, a0 = 2 mm: delta_K0 = 7.10, a_c = 99.1 mm, life about 1.39 million cycles (got {L.paris_closed_form(2e-3, ac80, 80.0):,.0f})",
                   abs(L.delta_k(80, 2e-3) - 7.10) < 0.005 and abs(ac80 * 1000 - 99.1) < 0.05 and abs(L.paris_closed_form(2e-3, ac80, 80.0) - 1388293) < 20))
    i05, i1, i2 = (L.inspection_interval_shifts(s, a_detect=a) for a in (0.5e-3, 1e-3, 2e-3))
    checks.append((f"Section F: detectable crack 0.5, 1, 2 mm gives intervals {i05:.0f}, {i1:.0f}, {i2:.0f} shifts", abs(i05 - 252.1) < 0.2 and abs(i1 - 129.1) < 0.2 and abs(i2 - 42.2) < 0.2))
    s7 = L.bracket_history(n_knocks=7)
    l7 = 1 / L.miner_damage(L.rainflow(s7))
    checks.append((f"mini-project: 7 knocks per shift give {l7:,.0f} shifts and an interval of {L.inspection_interval_shifts(s7):.0f} shifts", abs(l7 - 4310) < 5 and abs(L.inspection_interval_shifts(s7) - 233.7) < 0.2))
    checks.append(("app challenge: 60 MPa range, a0 = 2 mm, K_Ic = 50 gives at least 2 million cycles",
                   L.paris_closed_form(2e-3, L.critical_crack(60.0), 60.0) >= 2e6))
    checks.append(("Section F: 5th and 95th percentile lives are about 31,000 and 311,000 cycles; a 0.5 mm crack needs 3 x 252 = 756 shifts to become critical",
                   round(p5 / 1000) == 31 and round(p95 / 1000) == 311 and abs(3 * i05 - 756.3) < 0.5))
    checks.append(("the bracket history has 10 start-ups and 6 knocks (start-ups every 400 steps from step 250)", len(range(250, 4000 - 10, 400)) == 10 and abs(L.bracket_history(n_knocks=6).max() - 450) < 1))
    checks.append(("roadmap: the analytic 99 percent life is 0.2 of the median (10^(-2.326 x 0.3) = 0.2005)", abs(10 ** (-2.326 * 0.3) - 0.2005) < 0.0005))
    checks.append(("roadmap: the shipped K_t = 3 model gives about 326,000 cycles for the plate with nominal amplitude 110 MPa", abs(L.cycle_life(330) - 326247) < 20))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
